# Stage 2 (final) — DeepAR v1

Trains SageMaker's built-in **DeepAR** on the 333 valid clients, hourly resolution, forecasting
1 week (168 hours) ahead. This is the final, complete run of this notebook: baseline -> small HPO
-> GPU comparison -> batch transform evaluation -> a brief real-time endpoint lifecycle demo ->
persisted results for later comparison against LSTM/TFT.

**This run is meant to be executed once, start to finish, on this instance.** LSTM and TFT get
their own dedicated notebook instances later (see `setup_aws_resources.sh`) — this one's job ends
here.

**Cost note**: every job below is tagged (`Project`, `Stage`, `Model`) so spend is attributable in
Cost Explorer / Resource Explorer. The endpoint section deploys, tests once, and deletes
immediately — it does not stay up.

In [ ]:
%pip install -q duckdb "sagemaker<3" boto3

In [ ]:
import json
import time
from datetime import datetime, timezone
from pathlib import Path

import boto3
import duckdb
import numpy as np
import pandas as pd
import sagemaker
from sagemaker.estimator import Estimator
from sagemaker.image_uris import retrieve
from sagemaker.inputs import TrainingInput
from sagemaker.tuner import ContinuousParameter, HyperparameterTuner, IntegerParameter

REGION = boto3.Session().region_name or "us-east-1"
BUCKET = "<your-bucket>"
RAW_PREFIX = "ts-forecast-demo/raw/electricity"
CURATED_PREFIX = "ts-forecast-demo/curated"
DEEPAR_PREFIX = "ts-forecast-demo/deepar-v1"
RESULTS_PREFIX = "ts-forecast-demo/results/deepar-v1"
SAGEMAKER_ROLE = "arn:aws:iam::<ACCOUNT_ID>:role/ts-forecast-demo-sagemaker-role"

FREQ = "H"
PREDICTION_LENGTH = 24 * 7   # 1 week ahead
CONTEXT_LENGTH = 24 * 7      # 1 week of lookback (baseline default; tuned later)

TAGS = [
    {"Key": "Project", "Value": "ts-forecast-demo"},
    {"Key": "Stage", "Value": "2-deepar-v1"},
    {"Key": "Model", "Value": "deepar"},
]

session = sagemaker.Session()
s3 = boto3.client("s3")
sm = boto3.client("sagemaker")

## 1. Resample to hourly via DuckDB, filtered to valid clients

Same pushdown pattern as the EDA notebook: aggregation runs in DuckDB against S3, only the already-
hourly result comes into pandas.

In [ ]:
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
creds = session.boto_session.get_credentials().get_frozen_credentials()
con.execute(f"SET s3_region='{REGION}';")
con.execute(f"SET s3_access_key_id='{creds.access_key}';")
con.execute(f"SET s3_secret_access_key='{creds.secret_key}';")
if creds.token:
    con.execute(f"SET s3_session_token='{creds.token}';")

RAW_GLOB = f"s3://{BUCKET}/{RAW_PREFIX}/year=*/*.parquet"
METADATA_PATH = f"s3://{BUCKET}/{CURATED_PREFIX}/client_metadata.parquet"

hourly = con.sql(f"""
    SELECT
        r.client_id,
        time_bucket(INTERVAL '1 hour', r.timestamp) AS ts_hour,
        sum(r.kwh) AS kwh
    FROM read_parquet('{RAW_GLOB}', hive_partitioning=1) r
    JOIN read_parquet('{METADATA_PATH}') m USING (client_id)
    WHERE m.is_valid = true
    GROUP BY 1, 2
    ORDER BY 1, 2
""").df()

activation = con.sql(f"""
    SELECT client_id, first_active_ts
    FROM read_parquet('{METADATA_PATH}')
    WHERE is_valid = true
""").df().set_index("client_id")["first_active_ts"]

print(hourly.shape, hourly["client_id"].nunique())

## 2. Build DeepAR JSON Lines

Per-client series reindexed to a full hourly range from its own `first_active_ts`, gaps filled with
`"NaN"` (DeepAR's required missing-value token). `train.json` truncates the last
`PREDICTION_LENGTH` hours; `test.json` keeps the full series for scoring.

In [ ]:
def to_deepar_target(values: np.ndarray) -> list:
    return ["NaN" if pd.isna(v) else round(float(v), 4) for v in values]


client_ids = sorted(hourly["client_id"].unique())
client_index = {cid: i for i, cid in enumerate(client_ids)}

train_records, test_records = [], []

for cid, grp in hourly.groupby("client_id"):
    start = activation[cid].floor("h")
    full_range = pd.date_range(start, grp["ts_hour"].max(), freq="h")
    series = grp.set_index("ts_hour")["kwh"].reindex(full_range)

    record = {
        "start": start.strftime("%Y-%m-%d %H:%M:%S"),
        "target": to_deepar_target(series.values),
        "cat": [client_index[cid]],
    }
    test_records.append(record)

    train_record = dict(record)
    train_record["target"] = record["target"][:-PREDICTION_LENGTH]
    train_records.append(train_record)

print(f"{len(train_records)} series prepared")

In [ ]:
local_dir = Path("deepar_data")
local_dir.mkdir(exist_ok=True)

for name, records in [("train", train_records), ("test", test_records)]:
    path = local_dir / f"{name}.json"
    with open(path, "w") as f:
        for r in records:
            f.write(json.dumps(r) + "\n")
    s3.upload_file(str(path), BUCKET, f"{DEEPAR_PREFIX}/{name}/{name}.json")

train_s3 = f"s3://{BUCKET}/{DEEPAR_PREFIX}/train/"
test_s3 = f"s3://{BUCKET}/{DEEPAR_PREFIX}/test/"
print(train_s3, test_s3)

## 3. Baseline training job (CPU)

In [ ]:
deepar_image = retrieve("forecasting-deepar", REGION)

baseline_hyperparameters = {
    "time_freq": FREQ,
    "context_length": str(CONTEXT_LENGTH),
    "prediction_length": str(PREDICTION_LENGTH),
    "num_cells": "40",
    "num_layers": "2",
    "epochs": "30",
    "mini_batch_size": "64",
    "learning_rate": "1e-3",
    "likelihood": "gaussian",
    "early_stopping_patience": "10",
}

baseline_estimator = Estimator(
    image_uri=deepar_image,
    role=SAGEMAKER_ROLE,
    instance_type="ml.c5.xlarge",
    instance_count=1,
    output_path=f"s3://{BUCKET}/{DEEPAR_PREFIX}/model-baseline",
    hyperparameters=baseline_hyperparameters,
    sagemaker_session=session,
    tags=TAGS,
)

baseline_estimator.fit(
    {"train": TrainingInput(train_s3, content_type="json"), "test": TrainingInput(test_s3, content_type="json")}
)

## 4. Hyperparameter tuning (small budget, CPU)

6 jobs, 2 in parallel. Objective: `test:mean_wQuantileLoss`, lower is better.

In [ ]:
tuning_estimator = Estimator(
    image_uri=deepar_image,
    role=SAGEMAKER_ROLE,
    instance_type="ml.c5.xlarge",
    instance_count=1,
    output_path=f"s3://{BUCKET}/{DEEPAR_PREFIX}/model-hpo",
    hyperparameters={
        "time_freq": FREQ,
        "prediction_length": str(PREDICTION_LENGTH),
        "epochs": "30",
        "mini_batch_size": "64",
        "likelihood": "gaussian",
        "early_stopping_patience": "10",
    },
    sagemaker_session=session,
    tags=TAGS,
)

hyperparameter_ranges = {
    "context_length": IntegerParameter(24 * 3, 24 * 14),
    "num_cells": IntegerParameter(30, 100),
    "num_layers": IntegerParameter(1, 3),
    "learning_rate": ContinuousParameter(1e-4, 1e-2),
}

tuner = HyperparameterTuner(
    estimator=tuning_estimator,
    objective_metric_name="test:mean_wQuantileLoss",
    objective_type="Minimize",
    hyperparameter_ranges=hyperparameter_ranges,
    max_jobs=6,
    max_parallel_jobs=2,
    tags=TAGS,
)

tuner.fit(
    {"train": TrainingInput(train_s3, content_type="json"), "test": TrainingInput(test_s3, content_type="json")}
)

In [ ]:
best_job = tuner.best_training_job()
best_estimator = sagemaker.estimator.Estimator.attach(best_job, sagemaker_session=session)
best_hyperparameters = best_estimator.hyperparameters()
print(best_job)
print(best_hyperparameters)

## 5. GPU comparison run

Same best hyperparameters, different instance type -- comparing wall-clock/cost, not searching
for a better model.

In [ ]:
gpu_hyperparameters = {k: v for k, v in best_hyperparameters.items() if not k.startswith("_")}
gpu_hyperparameters["time_freq"] = FREQ
gpu_hyperparameters["prediction_length"] = str(PREDICTION_LENGTH)

gpu_estimator = Estimator(
    image_uri=deepar_image,
    role=SAGEMAKER_ROLE,
    instance_type="ml.g4dn.xlarge",
    instance_count=1,
    output_path=f"s3://{BUCKET}/{DEEPAR_PREFIX}/model-gpu",
    hyperparameters=gpu_hyperparameters,
    sagemaker_session=session,
    tags=TAGS,
)

gpu_estimator.fit(
    {"train": TrainingInput(train_s3, content_type="json"), "test": TrainingInput(test_s3, content_type="json")}
)

## 6. Evaluate via batch transform (not a persistent endpoint)

In [ ]:
transformer = best_estimator.transformer(
    instance_count=1,
    instance_type="ml.m5.large",
    output_path=f"s3://{BUCKET}/{DEEPAR_PREFIX}/batch-eval",
    strategy="SingleRecord",
    assemble_with="Line",
    accept="application/jsonlines",
    tags=TAGS,
)

transform_start = time.time()
transformer.transform(
    data=train_s3,
    content_type="application/jsonlines",
    split_type="Line",
)
transformer.wait()
transform_seconds = time.time() - transform_start

In [ ]:
output_key = f"{DEEPAR_PREFIX}/batch-eval/train.json.out"
obj = s3.get_object(Bucket=BUCKET, Key=output_key)
predictions = [json.loads(line) for line in obj["Body"].read().decode().splitlines()]

per_client_errors = []
for record, pred, cid in zip(test_records, predictions, client_ids):
    actual_tail = np.array([np.nan if v == "NaN" else v for v in record["target"][-PREDICTION_LENGTH:]])
    mean_pred = np.array(pred["quantiles"]["0.5"])
    mask = ~np.isnan(actual_tail)
    if mask.sum() > 0:
        rmse = np.sqrt(np.mean((actual_tail[mask] - mean_pred[mask]) ** 2))
        per_client_errors.append({"client_id": cid, "rmse": rmse})

errors_df = pd.DataFrame(per_client_errors)
print(f"Mean RMSE across {len(errors_df)} clients: {errors_df['rmse'].mean():.2f}")
print(f"Median RMSE: {errors_df['rmse'].median():.2f}")

## 7. Real-time endpoint — deploy, test once, delete

Demonstrates the full inference lifecycle for completeness. This is deliberately brief: deploy,
one test invocation, delete immediately. A persistent endpoint would be the wrong call for a
one-off demo and is exactly the kind of thing that quietly burns budget if left running.

In [ ]:
from sagemaker.deserializers import JSONDeserializer
from sagemaker.serializers import JSONSerializer

endpoint_start = time.time()
predictor = best_estimator.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large",
    endpoint_name="ts-forecast-demo-deepar-v1-test",
    serializer=JSONSerializer(),
    deserializer=JSONDeserializer(),
    tags=TAGS,
)

sample = train_records[0]
payload = {
    "instances": [{"start": sample["start"], "target": sample["target"], "cat": sample["cat"]}],
    "configuration": {"num_samples": 100, "output_types": ["mean", "quantiles"], "quantiles": ["0.1", "0.5", "0.9"]},
}
result = predictor.predict(payload)
print("Endpoint smoke test response (median forecast, first 5 hours):")
print(result["predictions"][0]["quantiles"]["0.5"][:5])

predictor.delete_endpoint(delete_endpoint_config=True)
endpoint_seconds = time.time() - endpoint_start
print(f"Endpoint deployed, tested, and deleted in {endpoint_seconds:.0f}s")

## 8. Persist results for later comparison (LSTM, TFT, Stage 3 registry)

Everything above lives in SageMaker's own job history indefinitely and could be re-queried later,
but the RMSE numbers only exist as this notebook's cell output. This writes a durable, structured
record to S3 so the LSTM and TFT notebooks (running on separate instances) can pull it in for a
fair side-by-side comparison, and so Stage 3's registry has a real artifact to formalize.

In [ ]:
def billable_seconds(job_name: str) -> int:
    return sm.describe_training_job(TrainingJobName=job_name)["TrainingTimeInSeconds"]


model_card = {
    "run_timestamp": datetime.now(timezone.utc).isoformat(),
    "stage": "2-deepar-v1",
    "n_clients": len(train_records),
    "freq": FREQ,
    "prediction_length": PREDICTION_LENGTH,
    "jobs": {
        "baseline": {
            "job_name": baseline_estimator.latest_training_job.name,
            "instance_type": "ml.c5.xlarge",
            "billable_seconds": billable_seconds(baseline_estimator.latest_training_job.name),
        },
        "best_hpo": {
            "job_name": best_job,
            "instance_type": "ml.c5.xlarge",
            "billable_seconds": billable_seconds(best_job),
            "hyperparameters": best_hyperparameters,
        },
        "gpu_comparison": {
            "job_name": gpu_estimator.latest_training_job.name,
            "instance_type": "ml.g4dn.xlarge",
            "billable_seconds": billable_seconds(gpu_estimator.latest_training_job.name),
        },
        "batch_transform": {
            "job_name": transformer.latest_transform_job.job_name,
            "instance_type": "ml.m5.large",
            "wall_clock_seconds": round(transform_seconds),
        },
        "endpoint_smoke_test": {
            "endpoint_name": "ts-forecast-demo-deepar-v1-test",
            "instance_type": "ml.m5.large",
            "wall_clock_seconds": round(endpoint_seconds),
            "deleted": True,
        },
    },
    "evaluation": {
        "metric": "RMSE (kWh), median forecast vs. held-out week",
        "mean_rmse": float(errors_df["rmse"].mean()),
        "median_rmse": float(errors_df["rmse"].median()),
        "n_scored_clients": len(errors_df),
    },
}

s3.put_object(
    Bucket=BUCKET,
    Key=f"{RESULTS_PREFIX}/model_card.json",
    Body=json.dumps(model_card, indent=2).encode(),
)

errors_local = local_dir / "deepar_v1_client_errors.parquet"
errors_df.to_parquet(errors_local, index=False)
s3.upload_file(str(errors_local), BUCKET, f"{RESULTS_PREFIX}/client_errors.parquet")

print(f"Persisted to s3://{BUCKET}/{RESULTS_PREFIX}/")
print(json.dumps(model_card, indent=2))

## Final check before closing this instance

- SageMaker console -> Endpoints: should be empty again (the smoke-test endpoint deleted itself
  in section 7).
- `model_card.json` and `client_errors.parquet` are now in S3 under `RESULTS_PREFIX` -- this is what
  the LSTM/TFT notebooks will read to build a fair comparison table.
- This notebook's job is done. Stop this instance; LSTM and TFT run on their own instances
  (see `setup_aws_resources.sh`).